# EDA tabular — Zonas Calientes (Nivel A: solicitudes + Nivel B: demanda agregada)

Exploración **en tabla** de los dos niveles del dataset sintético de demanda de viajes:
- **Nivel A** (`solicitudes_profesor.csv`): evento crudo por solicitud de viaje.
- **Nivel B** (`demanda_agregada_profesor.csv`): celda espacial × franja temporal × municipio.

**Objetivo:** conocer estructura, calidad y estadísticas antes de las visualizaciones y el clustering.

Referencias: `documentation/01-especificacion-dominio-y-esquema.md`, `documentation/03-diccionario-datos.md`


## 0. Carga de datos y configuración global

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Opciones de visualización (estilo del profe)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Rutas relativas al notebook (ubicado en notebooks/)
DATA_DIR = Path("..") / "documentation"
CSV_A    = DATA_DIR / "solicitudes_profesor.csv"
CSV_B    = DATA_DIR / "demanda_agregada_profesor.csv"
MANIFEST = DATA_DIR / "generation_manifest.json"

# Carga Nivel A
df_a = pd.read_csv(CSV_A, parse_dates=["timestamp"])

# Carga Nivel B
df_b = pd.read_csv(CSV_B)

print(f"Nivel A  cargado: {len(df_a):,} filas × {df_a.shape[1]} columnas")
print(f"Nivel B  cargado: {len(df_b):,} filas × {df_b.shape[1]} columnas")


Nivel A  cargado: 48,783 filas × 14 columnas
Nivel B  cargado: 5,357 filas × 14 columnas


## 1. Inventario de columnas

In [2]:
# --- Nivel A ---
BLOQUE_A = {
    "request_id":   "identificador",
    "timestamp":    "temporal",
    "municipio":    "geo",
    "lat":          "geo",
    "lng":          "geo",
    "hour":         "temporal",
    "day_of_week":  "temporal",
    "is_weekend":   "temporal",
    "is_holiday":   "temporal",
    "distancia_km": "viaje",
    "zona_destino": "viaje",
    "costo_mxn":    "viaje",
    "accepted":     "resultado",
    "zone_id_true": "ground_truth",
}

inventario_a = pd.DataFrame({
    "#": range(1, len(df_a.columns) + 1),
    "columna":    df_a.columns,
    "bloque":     [BLOQUE_A.get(c, "—") for c in df_a.columns],
    "dtype":      df_a.dtypes.astype(str).values,
    "no_nulos":   df_a.notna().sum().values,
    "nulos":      df_a.isna().sum().values,
    "pct_nulos":  (df_a.isna().mean() * 100).round(2).values,
    "unicos":     df_a.nunique(dropna=True).values,
})

print("=== Inventario Nivel A ===")
display(inventario_a)


=== Inventario Nivel A ===


,#,columna,bloque,dtype,no_nulos,nulos,pct_nulos,unicos
0,1,request_id,identificador,object,48783,0,0.0000,48300
1,2,timestamp,temporal,datetime64[ns],48783,0,0.0000,37331
2,3,municipio,geo,int64,48783,0,0.0000,2
3,4,lat,geo,float64,48783,0,0.0000,22277
4,5,lng,geo,float64,48783,0,0.0000,21903
5,6,hour,temporal,int64,48783,0,0.0000,30
6,7,day_of_week,temporal,int64,48783,0,0.0000,7
7,8,is_weekend,temporal,bool,48783,0,0.0000,2
8,9,is_holiday,temporal,bool,48783,0,0.0000,2
9,10,distancia_km,viaje,float64,48068,715,1.4700,1019


In [3]:
# --- Nivel B ---
BLOQUE_B = {
    "cell_id":              "identificador",
    "municipio":            "geo",
    "time_bucket":          "temporal",
    "dia_tipo":             "temporal",
    "hour":                 "temporal",
    "centroid_lat":         "geo",
    "centroid_lon":         "geo",
    "n_requests":           "demanda",
    "n_drivers_available":  "oferta",
    "supply_demand_ratio":  "oferta",
    "demand_density":       "demanda",
    "cancel_rate":          "resultado",
    "is_hot_true":          "ground_truth",
    "zone_id_true":         "ground_truth",
}

inventario_b = pd.DataFrame({
    "#": range(1, len(df_b.columns) + 1),
    "columna":    df_b.columns,
    "bloque":     [BLOQUE_B.get(c, "—") for c in df_b.columns],
    "dtype":      df_b.dtypes.astype(str).values,
    "no_nulos":   df_b.notna().sum().values,
    "nulos":      df_b.isna().sum().values,
    "pct_nulos":  (df_b.isna().mean() * 100).round(2).values,
    "unicos":     df_b.nunique(dropna=True).values,
})

print("=== Inventario Nivel B ===")
display(inventario_b)


=== Inventario Nivel B ===


,#,columna,bloque,dtype,no_nulos,nulos,pct_nulos,unicos
0,1,cell_id,identificador,object,5357,0,0.0000,1697
1,2,municipio,geo,int64,5357,0,0.0000,2
2,3,time_bucket,temporal,object,5357,0,0.0000,48
3,4,dia_tipo,temporal,object,5357,0,0.0000,2
4,5,hour,temporal,int64,5357,0,0.0000,24
5,6,centroid_lat,geo,float64,5357,0,0.0000,68
6,7,centroid_lon,geo,float64,5357,0,0.0000,60
7,8,n_requests,demanda,int64,5357,0,0.0000,165
8,9,n_drivers_available,oferta,int64,5357,0,0.0000,13
9,10,supply_demand_ratio,oferta,float64,5357,0,0.0000,249


## 2. Vista rápida (head / tail / sample)

In [4]:
print("=== Nivel A — primeras 5 filas ===")
display(df_a.head(5))

print("\n=== Nivel A — últimas 5 filas ===")
display(df_a.tail(5))

print("\n=== Nivel A — muestra aleatoria ===")
display(df_a.sample(5, random_state=42))


=== Nivel A — primeras 5 filas ===


,request_id,timestamp,municipio,lat,lng,hour,day_of_week,is_weekend,is_holiday,distancia_km,zona_destino,costo_mxn,accepted,zone_id_true
0,REQ-015726,2026-02-28 18:31:00,1,16.7583,-93.1294,18,5,True,False,NaN,0,12.0000,True,2
1,REQ-007259,2026-01-27 23:18:00,1,16.7808,-93.1469,23,1,False,False,3.9700,-1,20.0000,True,-1
2,REQ-038767,2026-02-24 09:34:00,2,16.9012,-92.0802,9,1,False,False,3.7200,1,16.0000,True,0
3,REQ-007290,2026-01-28 05:27:00,1,16.7515,-93.1147,5,2,False,False,3.1200,0,12.0000,True,0
4,REQ-030973,2026-01-26 08:36:00,2,16.8990,-92.0794,8,0,False,False,2.1700,0,14.0000,True,0



=== Nivel A — últimas 5 filas ===


,request_id,timestamp,municipio,lat,lng,hour,day_of_week,is_weekend,is_holiday,distancia_km,zona_destino,costo_mxn,accepted,zone_id_true
48778,REQ-042273,2026-03-09 13:09:00,2,16.9014,-92.0815,13,0,False,False,3.5800,0,14.0000,True,0
48779,REQ-043646,2026-03-14 20:20:00,2,16.9018,-92.0810,20,5,True,False,1.6300,-1,20.0000,True,0
48780,REQ-027203,2026-01-12 04:43:00,2,16.9090,-92.0695,4,0,False,False,1.3300,1,16.0000,True,1
48781,REQ-008482,2026-02-01 17:28:00,1,16.7516,-93.1165,17,6,True,False,3.8200,2,18.0000,True,0
48782,REQ-025855,2026-01-07 06:08:00,2,16.8994,-92.0814,6,2,False,False,1.7200,0,14.0000,True,0



=== Nivel A — muestra aleatoria ===


,request_id,timestamp,municipio,lat,lng,hour,day_of_week,is_weekend,is_holiday,distancia_km,zona_destino,costo_mxn,accepted,zone_id_true
47626,REQ-003710,2026-01-14 17:50:00,1,16.7507,-93.1176,17,2,False,False,1.1900,2,18.0000,True,0
47740,REQ-033497,2026-02-04 17:34:00,2,16.9090,-92.0699,17,2,False,False,1.9000,0,14.0000,True,1
27338,REQ-023436,2026-03-29 09:14:00,1,16.7347,-93.1108,9,6,True,False,4.0500,1,15.0000,True,1
46686,REQ-023875,2026-03-30 20:14:00,1,16.7362,-93.1124,20,0,False,False,1.6200,1,15.0000,False,1
13709,REQ-029124,2026-01-19 08:22:00,2,16.9104,-92.0684,8,0,False,False,2.2200,1,16.0000,True,1


In [5]:
print("=== Nivel B — primeras 5 filas ===")
display(df_b.head(5))

print("\n=== Nivel B — últimas 5 filas ===")
display(df_b.tail(5))

print("\n=== Nivel B — muestra aleatoria ===")
display(df_b.sample(5, random_state=42))


=== Nivel B — primeras 5 filas ===


,cell_id,municipio,time_bucket,dia_tipo,hour,centroid_lat,centroid_lon,n_requests,n_drivers_available,supply_demand_ratio,demand_density,cancel_rate,is_hot_true,zone_id_true
0,1-0-0,1,entre_semana_09,entre_semana,9,16.7013,-93.1587,1,2,2.0000,11.1100,0.0000,False,-1
1,1-0-0,1,entre_semana_17,entre_semana,17,16.7013,-93.1587,1,4,4.0000,11.1100,0.0000,False,-1
2,1-0-0,1,fin_de_semana_13,fin_de_semana,13,16.7013,-93.1587,1,7,7.0000,11.1100,1.0000,False,-1
3,1-0-2,1,entre_semana_07,entre_semana,7,16.7013,-93.1533,1,4,4.0000,11.1100,0.0000,False,-1
4,1-0-2,1,fin_de_semana_20,fin_de_semana,20,16.7013,-93.1533,1,3,3.0000,11.1100,0.0000,False,-1



=== Nivel B — últimas 5 filas ===


,cell_id,municipio,time_bucket,dia_tipo,hour,centroid_lat,centroid_lon,n_requests,n_drivers_available,supply_demand_ratio,demand_density,cancel_rate,is_hot_true,zone_id_true
5352,2-29-24,2,fin_de_semana_07,fin_de_semana,7,16.9396,-92.0539,1,4,4.0000,11.1100,0.0000,False,-1
5353,2-29-26,2,entre_semana_17,entre_semana,17,16.9396,-92.0485,1,7,7.0000,11.1100,1.0000,False,-1
5354,2-29-26,2,fin_de_semana_08,fin_de_semana,8,16.9396,-92.0485,1,7,7.0000,11.1100,0.0000,False,-1
5355,2-29-28,2,fin_de_semana_18,fin_de_semana,18,16.9396,-92.0431,1,1,1.0000,11.1100,0.0000,False,-1
5356,2-29-29,2,entre_semana_19,entre_semana,19,16.9396,-92.0404,1,7,7.0000,11.1100,0.0000,False,-1



=== Nivel B — muestra aleatoria ===


,cell_id,municipio,time_bucket,dia_tipo,hour,centroid_lat,centroid_lon,n_requests,n_drivers_available,supply_demand_ratio,demand_density,cancel_rate,is_hot_true,zone_id_true
4003,2-15-14,2,entre_semana_08,entre_semana,8,16.9018,-92.0809,180,3,0.0170,"2,000.0000",0.3500,True,0
1729,1-20-11,1,entre_semana_11,entre_semana,11,16.7553,-93.1290,1,6,6.0000,11.1100,0.0000,False,-1
401,1-8-10,1,entre_semana_17,entre_semana,17,16.7229,-93.1317,1,3,3.0000,11.1100,1.0000,False,-1
1242,1-16-15,1,entre_semana_15,entre_semana,15,16.7445,-93.1182,1,4,4.0000,11.1100,0.0000,False,-1
3406,2-10-14,2,entre_semana_20,entre_semana,20,16.8883,-92.0809,1,7,7.0000,11.1100,1.0000,False,-1


## 3. Resumen de nulos (Nivel A)

In [6]:
nulos_a = (
    df_a.isna()
    .sum()
    .rename("conteo")
    .to_frame()
    .assign(pct=lambda t: (t["conteo"] / len(df_a) * 100).round(2))
    .sort_values("conteo", ascending=False)
)
print("=== Nulos por columna — Nivel A ===")
display(nulos_a)

# Nulos por bloque
inv_a_tmp = inventario_a.copy()
inv_a_tmp.index = inv_a_tmp["columna"]
nulos_por_bloque = (
    inventario_a.groupby("bloque", as_index=False)
    .agg(
        columnas=("columna", "count"),
        nulos_totales=("nulos", "sum"),
        pct_promedio=("pct_nulos", "mean"),
    )
    .assign(pct_promedio=lambda t: t["pct_promedio"].round(2))
    .sort_values("nulos_totales", ascending=False)
)
print("\n=== Nulos por bloque — Nivel A ===")
display(nulos_por_bloque)


=== Nulos por columna — Nivel A ===


,conteo,pct
costo_mxn,749,1.5400
distancia_km,715,1.4700
municipio,0,0.0000
lat,0,0.0000
request_id,0,0.0000
timestamp,0,0.0000
hour,0,0.0000
lng,0,0.0000
is_weekend,0,0.0000
day_of_week,0,0.0000



=== Nulos por bloque — Nivel A ===


,bloque,columnas,nulos_totales,pct_promedio
5,viaje,3,1464,1.0000
0,geo,3,0,0.0000
1,ground_truth,1,0,0.0000
2,identificador,1,0,0.0000
3,resultado,1,0,0.0000
4,temporal,5,0,0.0000


In [7]:
# Nivel B — verificar nulos (no debería haber)
nulos_b = (
    df_b.isna()
    .sum()
    .rename("conteo")
    .to_frame()
    .assign(pct=lambda t: (t["conteo"] / len(df_b) * 100).round(2))
    .sort_values("conteo", ascending=False)
)
print("=== Nulos por columna — Nivel B (esperado: ninguno) ===")
display(nulos_b[nulos_b["conteo"] > 0] if (nulos_b["conteo"] > 0).any() else pd.DataFrame([{"resultado": "Sin nulos en Nivel B"}]))


=== Nulos por columna — Nivel B (esperado: ninguno) ===


,resultado
0,Sin nulos en Nivel B


## 4. Estadísticas numéricas (describe + CV)

In [8]:
NUM_A = ["distancia_km", "costo_mxn", "hour"]

desc_a = df_a[NUM_A].describe().T
desc_a["cv"] = (desc_a["std"] / desc_a["mean"]).replace([np.inf, -np.inf], np.nan).round(3)
print("=== Estadísticas Nivel A — numéricas principales ===")
display(desc_a)


=== Estadísticas Nivel A — numéricas principales ===


,count,mean,std,min,25%,50%,75%,max,cv
distancia_km,"48,068.0000",2.7801,1.4936,0.3400,1.7500,2.4500,3.4300,18.9700,0.5370
costo_mxn,"48,034.0000",15.7390,2.4837,12.0000,14.0000,16.0000,18.0000,20.0000,0.1580
hour,"48,783.0000",13.1880,6.3809,0.0000,8.0000,13.0000,19.0000,29.0000,0.4840


In [9]:
NUM_B = ["n_requests", "demand_density", "supply_demand_ratio", "cancel_rate"]

desc_b = df_b[NUM_B].describe().T
desc_b["cv"] = (desc_b["std"] / desc_b["mean"]).replace([np.inf, -np.inf], np.nan).round(3)
print("=== Estadísticas Nivel B — variables de demanda/oferta ===")
display(desc_b)


=== Estadísticas Nivel B — variables de demanda/oferta ===


,count,mean,std,min,25%,50%,75%,max,cv
n_requests,"5,357.0000",8.9246,29.1910,1.0000,1.0000,1.0000,2.0000,409.0000,3.2710
demand_density,"5,357.0000",99.1611,324.3446,11.1100,11.1100,11.1100,22.2200,"4,544.4400",3.2710
supply_demand_ratio,"5,357.0000",2.9786,2.2554,0.0000,1.0000,3.0000,5.0000,12.0000,0.7570
cancel_rate,"5,357.0000",0.2216,0.3710,0.0000,0.0000,0.0000,0.2880,1.0000,1.6740


## 5. Percentiles

In [10]:
PCTS = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
PCTS_LABELS = [f"p{int(p*100)}" for p in PCTS]

pct_a = df_a[NUM_A].quantile(PCTS).T
pct_a.columns = PCTS_LABELS
print("=== Percentiles Nivel A ===")
display(pct_a)


=== Percentiles Nivel A ===


,p1,p5,p25,p50,p75,p95,p99
distancia_km,0.7800,1.0800,1.7500,2.4500,3.4300,5.6200,7.9100
costo_mxn,12.0000,12.0000,14.0000,16.0000,18.0000,20.0000,20.0000
hour,0.0000,2.0000,8.0000,13.0000,19.0000,22.0000,23.0000


In [11]:
pct_b = df_b[NUM_B].quantile(PCTS).T
pct_b.columns = PCTS_LABELS
print("=== Percentiles Nivel B ===")
display(pct_b)


=== Percentiles Nivel B ===


,p1,p5,p25,p50,p75,p95,p99
n_requests,1.0000,1.0000,1.0000,1.0000,2.0000,43.0000,150.0800
demand_density,11.1100,11.1100,11.1100,11.1100,22.2200,477.7800,"1,667.5532"
supply_demand_ratio,0.0000,0.0470,1.0000,3.0000,5.0000,7.0000,9.0000
cancel_rate,0.0000,0.0000,0.0000,0.0000,0.2880,1.0000,1.0000


## 6. Frecuencias categóricas / booleanas

In [12]:
def tabla_frecuencia(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """Tabla de frecuencias con conteo y porcentaje para una columna."""
    vc = df[col].value_counts(dropna=False)
    out = vc.rename("conteo").to_frame().reset_index()
    out.columns = [col, "conteo"]
    out["pct"] = (out["conteo"] / len(df) * 100).round(2)
    return out

# Nivel A: categóricas y booleanas
FREQ_A = ["municipio", "is_weekend", "is_holiday", "accepted"]
print("=== Frecuencias Nivel A ===")
for col in FREQ_A:
    print(f"\n--- {col} ---")
    display(tabla_frecuencia(df_a, col))


=== Frecuencias Nivel A ===

--- municipio ---


,municipio,conteo,pct
0,1,24397,50.0100
1,2,24386,49.9900



--- is_weekend ---


,is_weekend,conteo,pct
0,False,33234,68.1300
1,True,15549,31.8700



--- is_holiday ---


,is_holiday,conteo,pct
0,False,46840,96.0200
1,True,1943,3.9800



--- accepted ---


,accepted,conteo,pct
0,True,37052,75.9500
1,False,11731,24.0500


In [13]:
# Nivel B: categóricas y booleanas
FREQ_B = ["municipio", "dia_tipo", "is_hot_true"]
print("=== Frecuencias Nivel B ===")
for col in FREQ_B:
    print(f"\n--- {col} ---")
    display(tabla_frecuencia(df_b, col))


=== Frecuencias Nivel B ===

--- municipio ---


,municipio,conteo,pct
0,1,2760,51.5200
1,2,2597,48.4800



--- dia_tipo ---


,dia_tipo,conteo,pct
0,entre_semana,3372,62.9500
1,fin_de_semana,1985,37.0500



--- is_hot_true ---


,is_hot_true,conteo,pct
0,False,4261,79.5400
1,True,1096,20.4600


## 7. Duplicados (Nivel A)

In [14]:
resumen_dup = pd.DataFrame({
    "metrica": [
        "filas_totales",
        "request_id_unicos",
        "request_id_duplicados",
        "filas_exactamente_duplicadas",
    ],
    "valor": [
        len(df_a),
        df_a["request_id"].nunique(),
        int(df_a["request_id"].duplicated().sum()),
        int(df_a.duplicated().sum()),
    ],
})
print("=== Resumen de duplicados — Nivel A ===")
display(resumen_dup)

# Mostrar algunos duplicados exactos si los hay
dup_mask = df_a.duplicated(keep=False)
n_dup = dup_mask.sum()
print(f"\nFilas en grupos duplicados: {n_dup:,}")
if n_dup > 0:
    display(df_a[dup_mask].sort_values("request_id").head(10))


=== Resumen de duplicados — Nivel A ===


,metrica,valor
0,filas_totales,48783
1,request_id_unicos,48300
2,request_id_duplicados,483
3,filas_exactamente_duplicadas,483



Filas en grupos duplicados: 966


,request_id,timestamp,municipio,lat,lng,hour,day_of_week,is_weekend,is_holiday,distancia_km,zona_destino,costo_mxn,accepted,zone_id_true
48308,REQ-000125,2026-01-01 09:03:00,1,16.7531,-93.1166,9,3,False,True,7.5500,2,18.0000,True,0
22393,REQ-000125,2026-01-01 09:03:00,1,16.7531,-93.1166,9,3,False,True,7.5500,2,18.0000,True,0
45458,REQ-000163,2026-01-01 13:58:00,1,16.7598,-93.1314,13,3,False,True,3.9900,1,15.0000,True,2
48357,REQ-000163,2026-01-01 13:58:00,1,16.7598,-93.1314,13,3,False,True,3.9900,1,15.0000,True,2
48572,REQ-000183,2026-01-01 16:08:00,1,16.7512,-93.1163,16,3,False,True,1.9200,0,12.0000,False,0
5908,REQ-000183,2026-01-01 16:08:00,1,16.7512,-93.1163,16,3,False,True,1.9200,0,12.0000,False,0
8875,REQ-000187,2026-01-01 16:56:00,1,16.7498,-93.1158,16,3,False,True,2.3600,1,15.0000,True,0
48642,REQ-000187,2026-01-01 16:56:00,1,16.7498,-93.1158,16,3,False,True,2.3600,1,15.0000,True,0
7646,REQ-000333,2026-01-02 02:35:00,1,16.7521,-93.1136,2,4,False,False,3.3900,1,15.0000,True,0
48356,REQ-000333,2026-01-02 02:35:00,1,16.7521,-93.1136,2,4,False,False,3.3900,1,15.0000,True,0


## 8. Reglas de calidad (Nivel A)

In [15]:
# BBox de referencia por municipio (del config de generación)
BBOX = {
    1: {"lat": [16.70, 16.80], "lng": [-93.16, -93.08]},
    2: {"lat": [16.86, 16.94], "lng": [-92.12, -92.04]},
}

def coords_fuera_bbox(df: pd.DataFrame) -> pd.Series:
    """Retorna máscara True donde las coordenadas están fuera del bbox del municipio."""
    fuera = pd.Series(True, index=df.index)  # inicia como 'no dentro de ningún bbox'
    for muni, bb in BBOX.items():
        m = df["municipio"] == muni
        dentro_muni = (
            m &
            df["lat"].between(bb["lat"][0], bb["lat"][1]) &
            df["lng"].between(bb["lng"][0], bb["lng"][1])
        )
        fuera = fuera & ~dentro_muni
    return fuera

reglas: dict[str, pd.Series] = {
    "hora_invalida (>23)":       df_a["hour"] > 23,
    "coords_fuera_bbox":         coords_fuera_bbox(df_a),
    "costo_negativo":            df_a["costo_mxn"].fillna(0) < 0,
    "distancia_negativa":        df_a["distancia_km"].fillna(0) < 0,
    "lat_nula":                  df_a["lat"].isna(),
    "lng_nula":                  df_a["lng"].isna(),
}

calidad = (
    pd.Series({k: int(v.sum()) for k, v in reglas.items()}, name="conteo")
    .to_frame()
    .assign(pct_del_total=lambda t: (t["conteo"] / len(df_a) * 100).round(3))
    .sort_values("conteo", ascending=False)
)
print("=== Violaciones de reglas de calidad — Nivel A ===")
display(calidad)


=== Violaciones de reglas de calidad — Nivel A ===


,conteo,pct_del_total
coords_fuera_bbox,490,1.0040
hora_invalida (>23),484,0.9920
costo_negativo,0,0.0000
distancia_negativa,0,0.0000
lat_nula,0,0.0000
lng_nula,0,0.0000


## 9. Segmentos

In [16]:
# Crosstab dia_semana × hora — conteo de solicitudes (Nivel A)
ct_demanda = pd.crosstab(
    df_a["day_of_week"],
    df_a["hour"],
    margins=True,
)
ct_demanda.index = [
    "Lun (0)", "Mar (1)", "Mie (2)", "Jue (3)", "Vie (4)",
    "Sab (5)", "Dom (6)", "Total",
]
print("=== Crosstab día_semana × hora — Nivel A (solicitudes) ===")
print("(solo horas con demanda)")
# Mostrar solo columnas con valores en el rango 0-23 + All
cols_validas = [c for c in ct_demanda.columns if isinstance(c, int) and 0 <= c <= 23] + ["All"]
display(ct_demanda[cols_validas])


=== Crosstab día_semana × hora — Nivel A (solicitudes) ===
(solo horas con demanda)


hour,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,All
Lun (0),134,137,133,132,134,135,134,682,679,541,131,132,137,412,132,131,132,677,687,599,461,136,135,136,6943
Mar (1),129,128,129,128,131,131,129,650,655,511,130,131,131,389,131,128,130,652,649,568,442,132,133,129,6665
Mie (2),119,119,120,122,119,120,121,596,598,486,117,121,116,362,121,119,121,605,605,530,406,121,119,118,6164
Jue (3),132,133,131,128,133,133,130,658,668,534,132,131,131,397,134,132,131,658,660,584,459,133,134,131,6790
Vie (4),132,130,130,129,129,130,131,649,649,520,132,130,129,391,131,131,129,648,649,569,441,130,132,130,6672
Sab (5),131,129,129,129,130,129,129,647,651,521,133,127,131,392,128,129,127,651,654,569,444,518,522,445,7770
Dom (6),131,131,129,131,132,129,132,658,648,521,130,129,130,392,130,130,130,645,652,571,440,519,521,439,7779
Total,908,907,901,899,908,907,906,4540,4548,3634,905,901,905,2735,907,900,900,4536,4556,3990,3093,1689,1696,1528,48783


In [17]:
# Top 5 horas más demandadas (Nivel A)
top_horas = df_a[df_a["hour"].between(0, 23)].groupby("hour").size().nlargest(5).reset_index()
top_horas.columns = ["hora", "solicitudes"]
print("=== Top 5 horas con mayor demanda — Nivel A ===")
display(top_horas)


=== Top 5 horas con mayor demanda — Nivel A ===


,hora,solicitudes
0,18,4556
1,8,4548
2,7,4540
3,17,4536
4,19,3990


In [18]:
# demand_density y supply_demand_ratio promedio por is_hot_true (Nivel B)
seg_hot = (
    df_b.groupby("is_hot_true", observed=True)
    .agg(
        n_celdas        = ("cell_id", "count"),
        demand_density_media  = ("demand_density", "mean"),
        demand_density_median = ("demand_density", "median"),
        supply_demand_media   = ("supply_demand_ratio", "mean"),
        supply_demand_median  = ("supply_demand_ratio", "median"),
        cancel_rate_media     = ("cancel_rate", "mean"),
    )
    .round(3)
)
print("=== Segmento is_hot_true — Nivel B ===")
display(seg_hot)


=== Segmento is_hot_true — Nivel B ===


,n_celdas,demand_density_media,demand_density_median,supply_demand_media,supply_demand_median,cancel_rate_media
is_hot_true,,,,,,
False,4261,13.4360,11.1100,3.6700,3.5000,0.2300
True,1096,432.4420,211.1100,0.2910,0.1820,0.1880


In [19]:
# Demanda media por dia_tipo y hora — Nivel B
pivot_demanda_b = df_b.pivot_table(
    index="dia_tipo",
    columns="hour",
    values="demand_density",
    aggfunc="mean",
).round(2)
print("=== demand_density media por dia_tipo × hora — Nivel B ===")
display(pivot_demanda_b)


=== demand_density media por dia_tipo × hora — Nivel B ===


hour,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
dia_tipo,,,,,,,,,,,,,,,,,,,,,,,,
entre_semana,81.8600,82.5600,70.9300,83.7300,73.7700,86.2100,88.7500,135.2800,125.7000,113.6400,76.5700,83.6600,83.0100,116.0300,82.4300,78.6400,91.0200,123.9200,131.8200,138.0800,111.3600,86.7500,82.7600,84.6600
fin_de_semana,59.1800,50.1000,69.3800,53.8800,55.7700,55.1300,49.8100,98.7700,98.1600,107.5800,57.0800,54.6800,45.1400,79.8300,57.1100,58.0500,59.3400,121.8400,108.2700,111.8000,86.4100,90.1100,100.9800,105.9200


## 10. Correlación numérica (Nivel B)

In [20]:
NUM_B_CORR = ["n_requests", "demand_density", "supply_demand_ratio", "cancel_rate"]
corr_b = df_b[NUM_B_CORR].corr(numeric_only=True).round(4)
print("=== Matriz de correlación — Nivel B ===")
display(corr_b)


=== Matriz de correlación — Nivel B ===


,n_requests,demand_density,supply_demand_ratio,cancel_rate
n_requests,1.0000,1.0000,-0.3433,0.0153
demand_density,1.0000,1.0000,-0.3433,0.0153
supply_demand_ratio,-0.3433,-0.3433,1.0000,0.0260
cancel_rate,0.0153,0.0153,0.0260,1.0000


In [21]:
# Top pares por |r| (triángulo superior)
triu_mask = np.triu(np.ones(corr_b.shape, dtype=bool), k=1)
stack = corr_b.where(triu_mask).stack()
stack.index.names = ["var_a", "var_b"]
top_pares = (
    stack.reset_index()
    .rename(columns={0: "r"})
    .assign(abs_r=lambda t: t["r"].abs())
    .sort_values("abs_r", ascending=False)
    .drop(columns="abs_r")
    .head(10)
    .reset_index(drop=True)
)
print("=== Top pares de correlación — Nivel B ===")
display(top_pares)


=== Top pares de correlación — Nivel B ===


,var_a,var_b,r
0,n_requests,demand_density,1.0000
1,n_requests,supply_demand_ratio,-0.3433
2,demand_density,supply_demand_ratio,-0.3433
3,supply_demand_ratio,cancel_rate,0.0260
4,n_requests,cancel_rate,0.0153
5,demand_density,cancel_rate,0.0153


## 11. Muestras estratificadas por `demand_density` (Nivel B)

In [22]:
COLS_VISTA_B = [
    "cell_id", "municipio", "dia_tipo", "hour",
    "n_requests", "demand_density", "supply_demand_ratio",
    "cancel_rate", "is_hot_true", "zone_id_true",
]

print("=== Top 10 celdas con mayor demand_density ===")
display(df_b.nlargest(10, "demand_density")[COLS_VISTA_B].reset_index(drop=True))


=== Top 10 celdas con mayor demand_density ===


,cell_id,municipio,dia_tipo,hour,n_requests,demand_density,supply_demand_ratio,cancel_rate,is_hot_true,zone_id_true
0,2-18-18,2,entre_semana,18,409,"4,544.4400",0.0170,0.3370,True,1
1,2-18-18,2,entre_semana,8,399,"4,433.3300",0.0130,0.3060,True,1
2,2-18-18,2,entre_semana,17,389,"4,322.2200",0.0100,0.3470,True,1
3,2-18-18,2,entre_semana,7,368,"4,088.8900",0.0050,0.3590,True,1
4,2-18-18,2,entre_semana,19,356,"3,955.5600",0.0140,0.2890,True,1
5,2-14-14,2,entre_semana,17,355,"3,944.4400",0.0060,0.3130,True,0
6,2-14-14,2,entre_semana,18,350,"3,888.8900",0.0090,0.3510,True,0
7,2-14-14,2,entre_semana,8,347,"3,855.5600",0.0200,0.3520,True,0
8,2-14-14,2,entre_semana,7,336,"3,733.3300",0.0150,0.3420,True,0
9,1-19-16,1,entre_semana,7,326,"3,622.2200",0.0030,0.3620,True,0


In [23]:
print("=== Bottom 10 celdas (menor demand_density) ===")
display(df_b.nsmallest(10, "demand_density")[COLS_VISTA_B].reset_index(drop=True))


=== Bottom 10 celdas (menor demand_density) ===


,cell_id,municipio,dia_tipo,hour,n_requests,demand_density,supply_demand_ratio,cancel_rate,is_hot_true,zone_id_true
0,1-0-0,1,entre_semana,9,1,11.1100,2.0000,0.0000,False,-1
1,1-0-0,1,entre_semana,17,1,11.1100,4.0000,0.0000,False,-1
2,1-0-0,1,fin_de_semana,13,1,11.1100,7.0000,1.0000,False,-1
3,1-0-2,1,entre_semana,7,1,11.1100,4.0000,0.0000,False,-1
4,1-0-2,1,fin_de_semana,20,1,11.1100,3.0000,0.0000,False,-1
5,1-0-3,1,entre_semana,10,1,11.1100,5.0000,0.0000,False,-1
6,1-0-4,1,entre_semana,18,1,11.1100,3.0000,1.0000,False,-1
7,1-0-5,1,entre_semana,10,1,11.1100,4.0000,0.0000,False,-1
8,1-0-5,1,entre_semana,20,1,11.1100,8.0000,0.0000,False,-1
9,1-0-5,1,fin_de_semana,6,1,11.1100,5.0000,0.0000,False,-1


In [24]:
print("=== Top 10 celdas con menor supply_demand_ratio (mayor presión demanda/oferta) ===")
display(df_b.nsmallest(10, "supply_demand_ratio")[COLS_VISTA_B].reset_index(drop=True))


=== Top 10 celdas con menor supply_demand_ratio (mayor presión demanda/oferta) ===


,cell_id,municipio,dia_tipo,hour,n_requests,demand_density,supply_demand_ratio,cancel_rate,is_hot_true,zone_id_true
0,1-0-20,1,fin_de_semana,22,1,11.1100,0.0000,0.0000,False,-1
1,1-0-26,1,entre_semana,14,1,11.1100,0.0000,0.0000,False,-1
2,1-2-12,1,entre_semana,10,1,11.1100,0.0000,0.0000,False,-1
3,1-2-13,1,entre_semana,1,1,11.1100,0.0000,0.0000,False,-1
4,1-2-25,1,entre_semana,9,1,11.1100,0.0000,0.0000,False,-1
5,1-4-21,1,entre_semana,9,1,11.1100,0.0000,0.0000,False,-1
6,1-7-18,1,fin_de_semana,6,1,11.1100,0.0000,0.0000,False,-1
7,1-8-0,1,fin_de_semana,8,1,11.1100,0.0000,0.0000,False,-1
8,1-8-20,1,fin_de_semana,19,1,11.1100,0.0000,1.0000,False,-1
9,1-8-22,1,fin_de_semana,19,1,11.1100,0.0000,1.0000,False,-1


In [25]:
# Muestra estratificada 2 por municipio x dia_tipo x is_hot_true
muestra_estratificada = (
    df_b.groupby(["municipio", "dia_tipo", "is_hot_true"], group_keys=False, observed=True)
    .apply(lambda g: g.sample(min(2, len(g)), random_state=42), include_groups=False)
    .reset_index(drop=True)
)
print(f"=== Muestra estratificada (municipio x dia_tipo x is_hot_true): {len(muestra_estratificada)} filas ===")
display(muestra_estratificada.head(20))


=== Muestra estratificada (municipio x dia_tipo x is_hot_true): 16 filas ===


,cell_id,time_bucket,hour,centroid_lat,centroid_lon,n_requests,n_drivers_available,supply_demand_ratio,demand_density,cancel_rate,zone_id_true
0,1-22-12,entre_semana_20,20,16.7607,-93.1263,1,3,3.0000,11.1100,0.0000,2
1,1-30-1,entre_semana_09,9,16.7823,-93.1560,1,1,1.0000,11.1100,1.0000,-1
2,1-18-15,entre_semana_09,9,16.7499,-93.1182,62,7,0.1130,688.8900,0.3230,0
3,1-13-17,entre_semana_08,8,16.7364,-93.1128,41,4,0.0980,455.5600,0.3900,1
4,1-18-15,fin_de_semana_03,3,16.7499,-93.1182,4,4,1.0000,44.4400,0.0000,0
5,1-14-25,fin_de_semana_13,13,16.7391,-93.0912,1,4,4.0000,11.1100,1.0000,-1
6,1-12-18,fin_de_semana_21,21,16.7337,-93.1101,11,2,0.1820,122.2200,0.2730,1
7,1-22-10,fin_de_semana_12,12,16.7607,-93.1317,6,1,0.1670,66.6700,0.0000,2
8,2-22-14,entre_semana_05,5,16.9207,-92.0809,1,5,5.0000,11.1100,0.0000,-1
9,2-19-12,entre_semana_20,20,16.9126,-92.0863,1,5,5.0000,11.1100,0.0000,-1


## 12. Manifiesto de generación

In [26]:
if MANIFEST.exists():
    with open(MANIFEST, encoding="utf-8") as f:
        manifest = json.load(f)
    manifiesto_df = pd.DataFrame(
        [(k, v) for k, v in manifest.items() if not isinstance(v, dict)],
        columns=["campo", "valor"],
    )
    print("=== Manifiesto general ===")
    display(manifiesto_df)

    if "validacion" in manifest:
        val_df = pd.DataFrame(
            list(manifest["validacion"].items()),
            columns=["metrica", "valor"],
        )
        print("\n=== Validación del generador ===")
        display(val_df)
else:
    print("No se encontró generation_manifest.json")


=== Manifiesto general ===


,campo,valor
0,seed,42
1,solicitudes,48783
2,celdas,5357
3,celdas_hot,1096



=== Validación del generador ===


,metrica,valor
0,solicitudes,48783
1,nulos,1464
2,duplicados,483
3,coords_fuera_bbox,490
4,hora_invalida,484


## 13. Resumen ejecutivo

In [27]:
# Celdas calientes
n_hot = int(df_b["is_hot_true"].sum())
pct_hot = n_hot / len(df_b) * 100

# Horas pico (top 3, solo horas válidas)
top3_horas = (
    df_a[df_a["hour"].between(0, 23)]
    .groupby("hour")
    .size()
    .nlargest(3)
    .index.tolist()
)

# Tasa de aceptación
tasa_accept = df_a["accepted"].mean() * 100

# Correlación demand_density ↔ supply_demand_ratio
r_dd_sdr = df_b[["demand_density", "supply_demand_ratio"]].corr().iloc[0, 1]

# Calidad
total_violaciones = int(calidad["conteo"].sum())

resumen_final = pd.DataFrame({
    "hallazgo": [
        "Nivel A — filas × columnas",
        "Nivel B — celdas × columnas",
        "Columnas con nulos (Nivel A)",
        "Duplicados exactos (Nivel A)",
        "Tasa de aceptación (Nivel A)",
        "Horas pico de demanda (Nivel A, top 3)",
        "Celdas hot=True (Nivel B)",
        "% celdas hot (Nivel B)",
        "demand_density media hot vs. no-hot",
        "supply_demand_ratio media hot vs. no-hot",
        "Correlación demand_density ↔ supply_demand_ratio",
        "Violaciones de calidad totales (Nivel A)",
    ],
    "valor": [
        f"{len(df_a):,} × {df_a.shape[1]}",
        f"{len(df_b):,} × {df_b.shape[1]}",
        int((df_a.isna().sum() > 0).sum()),
        int(df_a.duplicated().sum()),
        f"{tasa_accept:.1f}%",
        str(top3_horas),
        n_hot,
        f"{pct_hot:.1f}%",
        f"hot={df_b[df_b['is_hot_true']]['demand_density'].mean():.1f}  /  no-hot={df_b[~df_b['is_hot_true']]['demand_density'].mean():.1f}",
        f"hot={df_b[df_b['is_hot_true']]['supply_demand_ratio'].mean():.3f}  /  no-hot={df_b[~df_b['is_hot_true']]['supply_demand_ratio'].mean():.3f}",
        f"{r_dd_sdr:.4f}",
        total_violaciones,
    ],
})
print("=== Resumen ejecutivo EDA tabular ===")
display(resumen_final)


=== Resumen ejecutivo EDA tabular ===


,hallazgo,valor
0,Nivel A — filas × columnas,"48,783 × 14"
1,Nivel B — celdas × columnas,"5,357 × 14"
2,Columnas con nulos (Nivel A),2
3,Duplicados exactos (Nivel A),483
4,Tasa de aceptación (Nivel A),76.0%
5,"Horas pico de demanda (Nivel A, top 3)","[18, 8, 7]"
6,Celdas hot=True (Nivel B),1096
7,% celdas hot (Nivel B),20.5%
8,demand_density media hot vs. no-hot,hot=432.4 / no-hot=13.4
9,supply_demand_ratio media hot vs. no-hot,hot=0.291 / no-hot=3.670


## 14. Notas metodológicas

- **Nulos en Nivel A:** solo `distancia_km` y `costo_mxn` tienen nulos (inyectados sintéticamente, ~1.5 % cada uno). El resto de columnas está completo.
- **Duplicados:** el generador inyecta un 1 % de duplicados exactos; se detectan en la Sección 7.
- **Anomalías de calidad:** `hora_invalida` (>23) y `coords_fuera_bbox` son inyecciones intencionales del generador para simular datos sucios de producción.
- **Nivel B sin nulos:** la agregación produce un dataset limpio; todas las columnas son completas.
- **Horas pico:** 7–8 h y 17–19 h concentran el mayor volumen de solicitudes (demanda_horaria del config).
- **Celdas hot:** 1096 / 5357 ≈ 20 % son calientes; su `demand_density` media es ~30× mayor que las no-calientes.
- **Correlación demand_density ↔ supply_demand_ratio ≈ −0.34:** relación negativa moderada; a mayor demanda, menor oferta relativa (esperado).
- **ground_truth (`zone_id_true`, `is_hot_true`):** **no** deben usarse como features del clustering, solo para validación (ARI/NMI/precision-recall) en `clustering.ipynb`.
